# 03. Обучение и сравнение архитектур
Обучаем все 6 вариантов на одном и том же наборе данных с одинаковым бюджетом эпох/ранней остановки и сравниваем кривые обучения (гл. 3.2 отчёта: AdamW, KL-annealing, gradient clipping, early stopping).

In [14]:
import sys, pathlib
for cand in [pathlib.Path.cwd(), pathlib.Path.cwd().parent]:
    src = cand / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        break

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.random.seed(42)


## Источник данных
- `"synthetic"` - синтетический ряд с 3 скрытыми режимами (работает сразу, без файлов).
- `"csv_recession"` - реальный ценовой ряд (`data/raw/...csv`) + метка рецессии NBER/FRED `USREC` (`data/raw/USREC.csv`), выровненная по датам ценового ряда.

`RUN_CLASSIFICATION_TASK` - включает classification-метрики (только если есть содержательная бинарная разметка, т.е. для `csv_recession`).

`LOG_TRANSFORM` - **критично для реальных ценовых рядов, охватывающих много лет**: абсолютная цена акции может вырасти в десятки раз за десятилетия, из-за чего обычная первая разность (`y_t - y_{t-1}`) имеет нестационарную (растущую со временем) дисперсию. Поскольку нормализация считается только по train (ранний, "дешёвый" период), на val/test (поздний, "дорогой" период) это приводит к аномально большим нормализованным значениям и val_loss, в разы/десятки раз превышающему train_loss - это проблема препроцессинга, а не модели. `LOG_TRANSFORM=True` дифференцирует log(price) (логарифмическую доходность) вместо самой цены - она масштабно-инвариантна и не "уезжает" с ростом уровня цены. По умолчанию включается автоматически для `csv_recession` и выключена для `synthetic`.

In [ ]:
DATA_SOURCE = "csv_recession"          # "synthetic" | "csv_recession"
RUN_CLASSIFICATION_TASK = None      # None -> авто по DATA_SOURCE; True/False
LOG_TRANSFORM = None

PRICE_CSV_PATH = "../data/raw/aapl.csv"     # колонка с датой + числовые колонки (Close, ...)
PRICE_DATE_COL = "Date"
VALUE_COLS = None                             # None -> все числовые колонки автоматически
USREC_CSV_PATH = "../data/raw/USREC.csv"     # колонки observation_date/DATE, USREC (0/1), из FRED

SERIES_LENGTH = 3000   # только для synthetic
WINDOW_LEN = 48
EPOCHS = 20

from tsprep.config import DataConfig

if DATA_SOURCE == "synthetic":
    from tsprep.data_preprocessing.synthetic import generate_series
    series = generate_series(length=SERIES_LENGTH, n_features=1, n_regimes=3, anomaly_rate=0.01, seed=42)
    raw = series.values
    regime = series.regime          # 0/1/2 - скрытый режим
    is_anomaly = series.is_anomaly
    if RUN_CLASSIFICATION_TASK is None:
        RUN_CLASSIFICATION_TASK = False
    if LOG_TRANSFORM is None:
        LOG_TRANSFORM = False

elif DATA_SOURCE == "csv_recession":
    from tsprep.data_preprocessing.loaders import load_series_with_recession_labels
    bundle = load_series_with_recession_labels(
        price_path=PRICE_CSV_PATH, label_path=USREC_CSV_PATH,
        price_date_col=PRICE_DATE_COL, value_cols=VALUE_COLS,
        label_date_col="observation_date",   # для старого формата FRED (fredgraph.csv) поставьте "DATE"
    )
    raw = bundle["values"]
    regime = bundle["labels"]       # 0/1 - реальная метка рецессии NBER, выровненная по датам
    is_anomaly = None
    print("value columns:", bundle["value_cols"], "| positive (recession) rate:", regime.mean())
    if RUN_CLASSIFICATION_TASK is None:
        RUN_CLASSIFICATION_TASK = True
    if LOG_TRANSFORM is None:
        LOG_TRANSFORM = True

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

data_cfg = DataConfig(window_len=WINDOW_LEN, n_features=raw.shape[1], batch_size=64,
                       log_transform=LOG_TRANSFORM, train_frac=0.55, test_frac=0.3)
print("DATA_SOURCE:", DATA_SOURCE, "| n_features:", data_cfg.n_features,
      "| RUN_CLASSIFICATION_TASK:", RUN_CLASSIFICATION_TASK, "| LOG_TRANSFORM:", LOG_TRANSFORM)


value columns: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'] | positive (recession) rate: 0.08760651816415009
DATA_SOURCE: csv_recession | n_features: 6 | RUN_CLASSIFICATION_TASK: True | LOG_TRANSFORM: True


In [16]:
from tsprep.data_preprocessing.datasets import prepare_and_window, build_dataloaders

prepared = prepare_and_window(raw, data_cfg, regime=regime, is_anomaly=is_anomaly)
train_loader, val_loader, test_loader = build_dataloaders(prepared, data_cfg)
print("train/val/test windows:", prepared.train_windows.shape[0], prepared.val_windows.shape[0], prepared.test_windows.shape[0])


train/val/test windows: 3631 956 1960


In [ ]:
from tsprep.config import ModelConfig, TrainConfig, ALL_MODEL_NAMES
from tsprep.models.autoencoder import build_model
from tsprep.mlcore.trainer import train_model
import torch

models = {}
histories = {}

for name in ALL_MODEL_NAMES:
    torch.manual_seed(42)
    model_cfg = ModelConfig(model_name=name, latent_dim=16, hidden_dim=48,
                             tcn_levels=4, d_model=48, n_heads=4, n_attn_layers=2)
    train_cfg = TrainConfig(epochs=EPOCHS, early_stopping_patience=8, kl_anneal_epochs=10)
    model = build_model(model_cfg, n_features=data_cfg.n_features, window_len=WINDOW_LEN)
    print(f"\n=== training {name} ===")
    history = train_model(model, train_loader, val_loader, train_cfg, verbose=True)
    models[name] = model
    histories[name] = history


In [ ]:
from tsprep.evaluation.visualize import plot_loss_curves

for name, history in histories.items():
    fig = plot_loss_curves(history, title=name)


## Сохранение обученных моделей, историй и конфигурации данных
`DATA_SOURCE`/`RUN_CLASSIFICATION_TASK` сохраняются вместе с моделями, чтобы ноутбук 04 мог загрузить ТЕ ЖЕ данные и не рассинхронизироваться.

In [19]:
from datetime import date

today = date.today()
today_str = today.strftime("%Y-%m-%d")

import pickle, pathlib
out_dir = pathlib.Path(f"../results/{today_str}")
out_dir.mkdir(exist_ok=True)

for name, model in models.items():
    torch.save(model.state_dict(), out_dir / f"model_{name}.pt")

with open(out_dir / "histories.pkl", "wb") as f:
    pickle.dump({k: vars(v) for k, v in histories.items()}, f)

with open(out_dir / "run_config.pkl", "wb") as f:
    pickle.dump({
        "DATA_SOURCE": DATA_SOURCE, "RUN_CLASSIFICATION_TASK": RUN_CLASSIFICATION_TASK,
        "PRICE_CSV_PATH": PRICE_CSV_PATH, "PRICE_DATE_COL": PRICE_DATE_COL, "VALUE_COLS": VALUE_COLS,
        "USREC_CSV_PATH": USREC_CSV_PATH, "SERIES_LENGTH": SERIES_LENGTH,
        "WINDOW_LEN": WINDOW_LEN, "EPOCHS": EPOCHS, "LOG_TRANSFORM": LOG_TRANSFORM,
    }, f)

print("saved to", out_dir.resolve())


saved to E:\SSAU_code\НИР\results\2026-08-08
